# Insole demo

Six force-sensitive resistors under a right insole, an ESP32-S3 sampling them at 100 Hz, and a host pipeline that logs, segments, extracts features and classifies gait. This notebook runs the whole pipeline **without hardware**: the simulator sits behind the same seam as the board, every stage is exercised on simulated frames, and the real captures from the board are brought in beside them wherever they exist.

Three ideas run through the project and the notebook follows them in order:

- **Spec first.** The wire format is a document (`docs/frame_spec.md`); firmware, logger and simulator implement it and a codec test pins its rejection paths.
- **Sim-parallel.** Every stage has a simulator or a stub behind the same interface as the hardware, so the host side was built and tested before a board existed, and the real captures became the first non-circular test of everything the simulator was tuned against.
- **Fault-injected.** The simulator can drop frames, corrupt checksums and reboot mid-stream, so the logger and the streamer were hardened against a misbehaving link before one was seen.

Run all. Every random draw is seeded. Runtime and cell outputs are committed so the figures render on GitHub; re-running regenerates them identically.

In [1]:
# --- 0. Put the kernel at the repository root (clone and install on Colab) ---
import os, subprocess, sys, time
from pathlib import Path

T_START = time.time()
REPO_URL = "https://github.com/parsaileslamlou/Insole.git"
IN_COLAB = "google.colab" in sys.modules

def sh(*cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode:
        raise SystemExit(f"FAILED ({r.returncode}): {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r

if IN_COLAB:
    os.chdir("/content")
    import shutil; shutil.rmtree("/content/Insole", ignore_errors=True)
    sh("git", "clone", "-q", REPO_URL, "/content/Insole")
    os.chdir("/content/Insole")
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".[analysis,notebook]")
else:
    here = Path.cwd()
    os.chdir(next(p for p in (here, *here.parents) if (p / "insole" / "detector.py").exists()))
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from insole import detector as D, calibration as C, heatmap as H, gait_gen as G
from insole.features import cop_frame
from insole.representations import SHIPPED, LETTER, features_under, transform_frames
from insole.paths import DATA_REAL, DATA_SIM, MODELS, DOCS, FIGURES

SEED = 1
rng = np.random.default_rng(SEED)
os.makedirs("figures/demo", exist_ok=True)
print("repo:", os.getcwd())
print(sh("git", "log", "--oneline", "-1").stdout.strip())
print(f"python {sys.version.split()[0]}  numpy {np.__version__}  pandas {pd.__version__}")

repo: C:\Users\kingp\source\repos\Insole
f96ff56 README: separate the two s4-zero measurements, whole-capture and in-stance
python 3.12.2  numpy 2.5.1  pandas 3.0.5


## 1. The contract: the frame

**Why first:** three things must agree on the bytes -- firmware, host logger, simulator -- and a disagreement produces validation failures with no useful message. So the format is a document, `docs/frame_spec.md`, and the code follows it. One newline-terminated ASCII line per sample: `INS,SEQ,TS_US,S0..S5,CHECKSUM`, checksum = sum of the eight integers mod 256.

In [2]:
line = G.make_frame(41, 152300, [2048, 1900, 300, 150, 2200, 3000])
print("frame      :", line)
print("checksum   :", G.checksum(41, 152300, [2048, 1900, 300, 150, 2200, 3000]), "= (41 + 152300 + 2048 + 1900 + 300 + 150 + 2200 + 3000) % 256")
print("parse ok   :", G.parse_frame(line))
print("bit flip   :", G.parse_frame(line.replace("2048", "2049")), "   <- every gate but the checksum accepts it")
print("truncated  :", G.parse_frame(line[:-1]))
print("boot text  :", G.parse_frame("rst:0x1 (POWERON),boot:0x8 (SPI_FAST_FLASH_BOOT)"))

frame      : INS,41,152300,2048,1900,300,150,2200,3000,147
checksum   : 147 = (41 + 152300 + 2048 + 1900 + 300 + 150 + 2200 + 3000) % 256
parse ok   : ('ok', [41, 152300, 2048, 1900, 300, 150, 2200, 3000])
bit flip   : ('bad_checksum', None)    <- every gate but the checksum accepts it
truncated  : ('bad_checksum', None)
boot text  : ('malformed', None)


## 2. Generate: the simulator behind the board's seam

**Why:** `insole/gait_gen.py` emits exactly the frames the firmware emits, so the logger, the detector and the classifier were built and tested before the board existed. Five streams cover the failure buckets a walk-only fixture would miss (a fast cadence, a light short-stride shuffle, a dead heel channel, standing still), and a sixth carries injected link faults. Noise is seeded here so the numbers below reproduce.

In [3]:
STREAMS = [
    ("demo_walk",    ["--mode", "walk"]),
    ("demo_fast",    ["--mode", "walk", "--cycle", "0.6"]),
    ("demo_shuffle", ["--mode", "shuffle"]),
    ("demo_dropout", ["--mode", "walk", "--dropout"]),
    ("demo_stand",   ["--mode", "standing"]),
    ("demo_faulty",  ["--mode", "walk", "--drop-rate", "0.01", "--corrupt-rate", "0.005", "--reset-at", "30", "--fault-seed", "1"]),
]
for i, (stem, flags) in enumerate(STREAMS):
    r = sh(sys.executable, "-m", "insole.gait_gen", "--out", f"data/sim/{stem}.txt", "--noise-seed", str(SEED + i), *flags)
    print(r.stdout.strip())

wrote data/sim/demo_walk.txt: 6000 lines, 6000 frames generated


wrote data/sim/demo_fast.txt: 6000 lines, 6000 frames generated


wrote data/sim/demo_shuffle.txt: 6000 lines, 6000 frames generated


wrote data/sim/demo_dropout.txt: 6000 lines, 6000 frames generated


wrote data/sim/demo_stand.txt: 6000 lines, 6000 frames generated


wrote data/sim/demo_faulty.txt: 5938 lines, 6000 frames generated, drop_rate=0.01, corrupt_rate=0.005, reset_at=30s, fault_seed=1


## 3. Log: the validator and its counters

**Why:** the logger is the one place bytes become rows. It parses, checks the checksum, tracks sequence and timing, counts every rejection, and exits nonzero on anything that would silently ruin a dataset. The same `FrameValidator` runs inside the streamer, so the two can never disagree about a frame. The faulty stream shows the counters doing their job: dropped frames are `lost`, corrupt ones `bad_checksum`, a reboot is `resets`, and the exit code is 1.

In [4]:
for stem in ("demo_walk", "demo_faulty"):
    r = sh(sys.executable, "-m", "insole.read_serial", f"data/sim/{stem}.txt", f"data/sim/{stem}.csv", check=False)
    print(f"{stem}: exit {r.returncode}")
    for ln in r.stdout.strip().splitlines():
        if ln.startswith(("source=", "FAIL", "reset")):
            print("   ", ln)

demo_walk: exit 0
    source=file valid=6000 malformed=0 empty=0 bad_checksum=0 seq_breaks=0 lost=0 loss=0.00% timing_breaks=0 resets=0 status=0 source_drops=0 capture_s=0.0 device_s=60.0


demo_faulty: exit 1
    reset: board rebooted at host t=0.0s (SEQ and ts_us restarted); rows continue in the same file
    source=file valid=5907 malformed=1 empty=0 bad_checksum=29 seq_breaks=63 lost=64 loss=1.07% timing_breaks=0 resets=1 status=1 source_drops=0 capture_s=0.0 device_s=30.0
    FAIL: corrupted frames present


## 4. Calibrate: a single-point relative gain match

**Why:** the six FSRs do not read the same count at the same force. What ships is a *single-point relative gain match* (`models/gain_match.json`): every channel pressed to ~12 N after a ≥35 min rest, its gain matched to the six-channel mean, applied in conductance space x = counts / (4095 − counts) where force is linear. It is **not** an absolute force calibration. The bench found the reason: FSRs relax ~31 % in 76 s under constant load and recover over ~20 min, so a multi-point absolute fit would have been fitting a clock (`docs/calibration_notes.md`).

Three documented limitations: anchored at one load on a nonlinear response; the working range in walking (peaks ~1900 counts) is above every bench sample (max 824 counts); and below ~5 N the channels' activation thresholds diverge (s4 read 0 counts at 2.58 N while s5 read 239 at 2.49 N). The extrapolation fraction below is the second limitation measured on the real captures.

In [5]:
gm = C.load_gain_match()
print("kind:", gm["kind"], "| fs_counts:", gm["fs_counts"], "| model:", gm["model"])
print("k = F/x per channel (N):", {k: round(v, 2) for k, v in gm["k"].items()}, " mean", round(gm["k_mean"], 2))
print("corrections k_i / mean :", {k: round(v, 4) for k, v in gm["corrections"].items()})
print(f"\nextrapolation: fraction of frames with any sensor above calibration.CAL_MAX_COUNTS = {C.CAL_MAX_COUNTS}")
for label, fname in [("stand", "stand_02.csv"), ("walk", "walk02.csv"), ("fast", "fast02.csv"), ("shuffle", "shuffle02.csv")]:
    df = pd.read_csv(DATA_REAL / fname)
    frac = (df[D.SENSOR_COLS].to_numpy() > C.CAL_MAX_COUNTS).any(axis=1).mean()
    print(f"  {label:8s} {frac:7.2%}   (scripts/analyze_real.py C3)")

kind: relative_gain_match | fs_counts: 4095.0 | model: corrected_x = correction[i] * (counts / (fs_counts - counts))
k = F/x per channel (N): {'0': 59.55, '1': 57.84, '2': 58.59, '3': 75.27, '4': 52.28, '5': 57.37}  mean 60.15
corrections k_i / mean : {0: 0.99, 1: 0.9616, 2: 0.9741, 3: 1.2513, 4: 0.8692, 5: 0.9538}

extrapolation: fraction of frames with any sensor above calibration.CAL_MAX_COUNTS = 824
  stand    100.00%   (scripts/analyze_real.py C3)
  walk      66.58%   (scripts/analyze_real.py C3)
  fast      62.45%   (scripts/analyze_real.py C3)
  shuffle   61.97%   (scripts/analyze_real.py C3)


## 5. Segment: stance detection on total force

**Why:** everything downstream is per stance. `insole/detector.py` is a two-threshold hysteresis state machine (T_ON to enter, a lower T_OFF to leave) with a minimum duration to reject spikes, a maximum duration to reject standing, and a merge for fragmented contacts. Four of the five constants were swept against the simulator; `MAX_DURATION` was set from real data after the simulator-derived 120 discarded 17 of 35 real walk contacts and 28 of 30 shuffles outright, because real contacts run 84–164 frames while no simulated stance exceeds 60.

In [6]:
def stances_of(csv_path):
    df = pd.read_csv(csv_path)
    total = df[D.SENSOR_COLS].sum(axis=1).to_numpy(dtype=float)
    return df, D.merge_close(D.find_stances(total))

sim_df, sim_st = stances_of("data/sim/demo_walk.csv")
real_df, real_st = stances_of(DATA_REAL / "walk02.csv")
print(f"thresholds: T_ON={D.T_ON} T_OFF={D.T_OFF} MIN_DURATION={D.MIN_DURATION} MAX_DURATION={D.MAX_DURATION} GAP_MERGE={D.GAP_MERGE}")
print(f"sim walk : {len(sim_st)} stances detected, {len(G.true_stances(60))} in the generator's truth, longest {max(b - a + 1 for a, b in sim_st)} frames")
print(f"real walk: {len(real_st)} stances detected, longest {max(b - a + 1 for a, b in real_st)} frames")
at120 = D.merge_close(D.find_stances(real_df[D.SENSOR_COLS].sum(axis=1).to_numpy(dtype=float), max_duration=120))
print(f"real walk at the old MAX_DURATION=120: {len(at120)} stances (over-ceiling runs are discarded, not clipped; scripts/sweep_max_duration.py)")

thresholds: T_ON=1200 T_OFF=450 MIN_DURATION=15 MAX_DURATION=200 GAP_MERGE=12
sim walk : 60 stances detected, 60 in the generator's truth, longest 58 frames
real walk: 35 stances detected, longest 144 frames
real walk at the old MAX_DURATION=120: 18 stances (over-ceiling runs are discarded, not clipped; scripts/sweep_max_duration.py)


## 6. Features: seven numbers per stance

**Why:** the classifier sees per-stance features, not frames. Five describe the total-force trace (peak, time to peak, contact time, loading rate, impulse) and two describe the centre-of-pressure path (its length and its end-to-end displacement, in normalised insole units; multiply by 274 for mm). Since stage 20 the extractors run on **representation B, conductance** (`insole/representations.py`), chosen on the real captures; the detector above still saw raw counts.

In [7]:
sim_feat = features_under(sim_df, sim_st, "sim_walk", SHIPPED)
real_feat = features_under(real_df, real_st, "real_walk", SHIPPED)
FEATS = ["peak_counts", "time_to_peak_s", "contact_time_s", "loading_rate_cps", "impulse_counts_s", "cop_path_len", "cop_displacement"]
print(f"features under representation {LETTER[SHIPPED]} ({SHIPPED}); count-valued columns are in conductance units")
summary = pd.concat({"sim walk (n=%d)" % len(sim_feat): sim_feat[FEATS].describe().loc[["mean", "std", "min", "max"]],
                     "real walk02 (n=%d)" % len(real_feat): real_feat[FEATS].describe().loc[["mean", "std", "min", "max"]]})
pd.set_option("display.width", 160); pd.set_option("display.float_format", lambda v: f"{v:.4f}")
summary

features under representation B (conductance); count-valued columns are in conductance units


peak_counts  time_to_peak_s  contact_time_s  loading_rate_cps  impulse_counts_s  cop_path_len  cop_displacement
sim walk (n=60)    mean       4.8955          0.4008          0.5603           11.0258            1.2278        0.9146            0.7151
                   std        0.0575          0.0056          0.0018            0.2204            0.0039        0.0168            0.0141
                   min        4.7694          0.3900          0.5600           10.5274            1.2143        0.8757            0.6771
                   max        5.0188          0.4100          0.5700           11.3969            1.2354        0.9491            0.7376
real walk02 (n=35) mean       1.8965          0.6406          1.1857            2.5829            1.6588        0.9280            0.5006
                   std        0.1389          0.2342          0.1388            1.3818            0.2268        0.2400            0.1905
                   min        1.5392          0.0500          0.8300            0.1411            1.0935        0.4243            0.0347
                   max        2.1311          1.0000          1.4300            7.7138            2.2205        1.7194            0.6727

## 7. Predict: from-scratch LDA and QDA

**Why two numbers:** the simulated bake-off (`scripts/bakeoff.py`) is a session-disjoint split over 12 simulated sessions, and it is internally consistent by construction -- the generator's constants, the detector thresholds and the tests were co-evolved. The real number (`scripts/train_real.py`, `docs/real_results.md`) is leave-one-session-out over two sessions per class: every stance is tested out of its own session, so nothing in training shares a session, a day or a sensor state with the stance being scored. Both are quoted from the scripts that compute them, never typed.

**Read the intervals as floors.** Every pooled interval printed below is a Wilson interval computed as if the 224 pooled stances were 224 independent observations. They are not: they are 224 stances from 2 sessions of 1 subject, and stances within a session share the subject, the day, the shoe, the sensor seating and the path. Treating correlated observations as independent understates the variance, so the true interval is **wider** than the one printed, by an amount two sessions cannot estimate. The per-fold intervals are the widest honest statement available. The output below carries the full caveat verbatim from the document rather than a summary of it.

**And note which representation the rule picked.** On this data the pre-registered rule prefers A (raw counts) over B (conductance) by two stances in 224 -- a margin finer than the data can resolve, with intervals that overlap over 93 % of their length. B is still what ships, retained as a pre-existing freeze taken at stage 20, **not** because the rule chose it. Real models are persisted under both, each labelled with its own `representation`, and `insole/infer_live.py` applies whichever one a model names.


In [8]:
r = sh(sys.executable, "scripts/bakeoff.py")
print("scripts/bakeoff.py (session-disjoint, 270 held-out simulated stances):")
for ln in r.stdout.splitlines():
    if "test accuracy" in ln or "MAJORITY-CLASS FLOOR" in ln or "feature representation" in ln:
        print("  ", ln.strip())

# Quoted whole from the document the script writes, never typed here and never
# truncated. The prefixes name the paragraphs that have to travel WITH the
# numbers: what the rule actually picked and why B is still shipped, that the
# pooled interval is a floor on the uncertainty rather than its extent, and that
# the two headline accuracies are on different denominators. A number lifted out
# of those paragraphs claims something the paragraphs say it does not.
import textwrap

WANT = ("Rule, fixed before any result",
        "**Shipped representation:",
        "**The pooled interval is a LOWER BOUND",
        "**Both representations are persisted",
        "**The two headline numbers are on different denominators",
        "The best full-feature cell",
        "Per-class recall")

doc = (DOCS / "real_results.md").read_text(encoding="utf-8").splitlines()
print()
print("docs/real_results.md (regenerate with: python scripts/train_real.py):")
seen = set()
for ln in doc:
    for w in WANT:
        if ln.startswith(w) and w not in seen:
            seen.add(w)
            print()
            print(textwrap.fill(ln, 100, initial_indent="   ", subsequent_indent="   "))

missing = [w for w in WANT if w not in seen]
if missing:
    # The document is regenerated; a paragraph that moves must not vanish silently.
    print()
    print("   MISSING from docs/real_results.md -- a paragraph was renamed or dropped:")
    for w in missing:
        print("     ", w)

scripts/bakeoff.py (session-disjoint, 270 held-out simulated stances):
   feature representation: B (conductance), insole.representations.SHIPPED
   MAJORITY-CLASS FLOOR (test set): always predict 'shuffle'
   LogisticRegression (scaled)   test accuracy = 0.9185   (248 / 270)   lift over floor = +0.4889
   my LDA                        test accuracy = 0.9185   (248 / 270)   lift over floor = +0.4889
   my QDA                        test accuracy = 0.9296   (251 / 270)   lift over floor = +0.5000

docs/real_results.md (regenerate with: python scripts/train_real.py):

   **Shipped representation: B (conductance)** -- `insole.representations.SHIPPED`, the one
   `insole/infer_live.py` feeds on every source, `scripts/bakeoff.py` builds the sim frame under,
   and the persisted models are fitted on. It was chosen at stage 20 by the headline rule in section
   4 on the `_02` set. On the current data the rule prefers A (raw) over B (conductance) by 2
   stance(s) in 224 -- 0.9% -- (pooled lea

## 8. Stream: one frame at a time

**Why:** the batch pipeline reads a whole file; a board delivers frames one at a time and never says when a stance ends. `insole/infer_live.py` runs the identical state machine incrementally (`StanceTracker`), computes the identical features the moment a stance closes, and the test suite asserts the two paths agree bit for bit on every capture in the repository. Switching from a file to USB or BLE is an argument. The default model is the one trained on the real captures; this cell replays a *simulated* walk, so it names the sim-trained model explicitly, the one that was fitted on frames like these.

In [9]:
r = sh(sys.executable, "-m", "insole.infer_live", "data/sim/demo_walk.txt", "--model", "models/model_lda.json", "--label", "walk", check=False)
lines = r.stdout.splitlines()
shown = 0
for ln in lines:
    if ln.startswith("stance") and shown < 5:
        print(ln[:118]); shown += 1
    elif ln.startswith(("model ", "features ", "stances completed", "predictions", "agreement")):
        print(ln)
print(f"exit {r.returncode}")

model     : models/model_lda.json  kind=lda  classes=['fast', 'shuffle', 'walk']  features=['cop_path_len', 'cop_displacement']
features  : representation B (conductance), read from the model's meta and applied on every source; the detector sees raw counts
stance    1  frames      4..    60 ( 57 fr, 0.56 s) t=   0.04s  path=0.9012 disp=0.7211  -> walk     gap_frames=0 | va
stance    2  frames    104..   160 ( 57 fr, 0.56 s) t=   1.04s  path=0.9114 disp=0.7271  -> walk     gap_frames=0 | va
stance    3  frames    204..   260 ( 57 fr, 0.56 s) t=   2.04s  path=0.9080 disp=0.7326  -> walk     gap_frames=0 | va
stance    4  frames    304..   360 ( 57 fr, 0.56 s) t=   3.04s  path=0.9157 disp=0.7242  -> walk     gap_frames=0 | va
stance    5  frames    404..   460 ( 57 fr, 0.56 s) t=   4.04s  path=0.8757 disp=0.7354  -> fast     gap_frames=0 | va
stances completed=60 discarded>MAX_DURATION(200)=0 discarded@reset=0 rejected<MIN_DURATION(15)=0 predicted=60 no_prediction=0 stances_with_gaps=0
pr

## 9. The pressure heatmap: simulated walk beside a real one

**Why:** the six readings and the CoP path are easier to believe when you can watch them. The field between the sensors is inverse-distance-weighted interpolation across six points -- **6 sensors; values between them are interpolated, not measured** -- and the render says so. Each panel keeps a fixed colour scale across its frames (per panel: the two stances differ in magnitude by a factor of a few), the colourbar is in the units on display, conductance x = counts / (4095 − counts), which is what the classifier sees, and the green marker with its trail is the CoP under that same representation.

**Playback ratio:** a 100 Hz stance cannot play at 100 fps in a GIF (browsers clamp GIF frame delays near 100 ms). Every third frame is kept (an effective 33 Hz) and the GIF plays at 10 frames per second, so **one GIF second is 0.3 s of real time -- playback at 0.3× real time**. The frames are rendered at 72 dpi to keep the files small.

In [10]:
from matplotlib.animation import FuncAnimation, PillowWriter

STEP, FPS, DPI = 3, 10, 72   # every 3rd frame at 10 fps -> 0.3x real time; 72 dpi keeps the GIFs small
UNIT = "conductance x = c / (4095 − c)   (representation B)"

def stance_arrays(df, stance):
    a, b = stance
    seg = df.iloc[a:b]                                   # features.py slices [start:end]
    vals = transform_frames(seg[D.SENSOR_COLS].to_numpy(dtype=float), SHIPPED)
    cop = np.array([cop_frame(dict(zip(D.SENSOR_COLS, v))) for v in vals], dtype=float)
    ts = seg["ts_us"].to_numpy(dtype=float)
    return vals, cop, (ts - ts[0]) / 1e6

def animate_stance(df, stance, title, out):
    vals, cop, t_rel = stance_arrays(df, stance)
    vmax = float(vals.max())
    fig, ax = plt.subplots(figsize=(4.0, 7.2))
    im = ax.imshow(H.field(vals[0]), origin="lower", extent=H.EXTENT, cmap="magma",
                   vmin=0, vmax=vmax, interpolation="bilinear")
    H._draw_foot(ax)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label(UNIT, fontsize=7)
    fig.text(0.5, 0.015, H.CAPTION, ha="center", fontsize=6.5, color="0.35")
    trail, = ax.plot([], [], color="#39FF9E", lw=1.4, alpha=0.9, zorder=6)
    head, = ax.plot([], [], "o", color="#39FF9E", ms=7, mec="black", mew=0.8, zorder=7)
    frames = list(range(0, len(vals), STEP))
    def update(k):
        im.set_data(H.field(vals[k]))
        ax.set_title(f"{title}\nt = {t_rel[k]:.2f} s  frame {k + 1}/{len(vals)}  (0.3× real time)", fontsize=9)
        good = ~np.isnan(cop[:k + 1, 0])
        trail.set_data(cop[:k + 1, 0][good], cop[:k + 1, 1][good])
        if not np.isnan(cop[k, 0]):
            head.set_data([cop[k, 0]], [cop[k, 1]])
        return im, trail, head
    anim = FuncAnimation(fig, update, frames=frames, blit=False)
    anim.save(out, writer=PillowWriter(fps=FPS), dpi=DPI)
    plt.close(fig)
    return len(frames), cop

K = 5                                                     # the sixth stance of each capture
sim_stance, real_stance = sim_st[K], real_st[K]
n_sim, sim_cop = animate_stance(sim_df, sim_stance, "simulated walk (demo_walk)", "figures/demo/heatmap_sim_walk.gif")
n_real, real_cop = animate_stance(real_df, real_stance, "real walk (data/real/walk02.csv)", "figures/demo/heatmap_real_walk.gif")
for name, st, n in (("sim", sim_stance, n_sim), ("real", real_stance, n_real)):
    print(f"{name:4s} stance frames {st[0]}..{st[1]} ({st[1] - st[0]} shown, {(st[1] - st[0]) / 100:.2f} s real) -> {n} GIF frames at {FPS} fps = {n / FPS:.1f} s of GIF")
print("sizes:", {f: f"{os.path.getsize('figures/demo/' + f) / 1024:.0f} kB" for f in os.listdir("figures/demo")})

sim  stance frames 504..560 (56 shown, 0.56 s real) -> 19 GIF frames at 10 fps = 1.9 s of GIF
real stance frames 765..873 (108 shown, 1.08 s real) -> 36 GIF frames at 10 fps = 3.6 s of GIF
sizes: {'heatmap_real_walk.gif': '957 kB', 'heatmap_sim_walk.gif': '416 kB'}


![simulated walk](../figures/demo/heatmap_sim_walk.gif) ![real walk](../figures/demo/heatmap_real_walk.gif)

### Numeric agreement: the overlay is the feature

**Why:** a plot that disagrees with the number the classifier sees is worse than no plot. The CoP trail above is computed from the displayed frames; `features.py` computes `cop_displacement` for the same stance through its own path (`extract_features` → `cop_trajectory` → `cop_features`). The two must be equal to the last bit, and this cell asserts it. A mismatch here is a bug in one of them.

In [11]:
for name, df, st, cop in (("sim", sim_df, sim_stance, sim_cop), ("real", real_df, real_stance, real_cop)):
    valid = cop[~np.isnan(cop[:, 0])]
    overlay = float(np.hypot(valid[-1, 0] - valid[0, 0], valid[-1, 1] - valid[0, 1]))
    feature = float(features_under(df, [st], name, SHIPPED)["cop_displacement"].iloc[0])
    print(f"{name:4s} overlay displacement {overlay:.12f}   features.cop_displacement {feature:.12f}   "
          f"({overlay * D.INSOLE_LEN_MM:.1f} mm)")
    assert overlay == feature, (name, overlay, feature)
print("agreement: exact")

sim  overlay displacement 0.710851459488   features.cop_displacement 0.710851459488   (194.8 mm)
real overlay displacement 0.034660111820   features.cop_displacement 0.034660111820   (9.5 mm)
agreement: exact


## 10. Faults through the streamer

**Why:** the link will misbehave. The faulty stream from section 2 drops 1 % of frames, corrupts 0.5 % of checksums and reboots the board at 30 s. The streamer counts each, flags every stance whose span has a hole in it (`gap_frames`), discards the stance in progress at the reboot, re-seeds its clocks, and exits 1 -- the same counters the logger printed in section 3.

In [12]:
r = sh(sys.executable, "-m", "insole.infer_live", "data/sim/demo_faulty.txt", "--label", "walk", check=False)
for ln in r.stdout.splitlines():
    if ln.startswith(("reset", "discard")) or ln.startswith(("source=", "stances completed", "FAIL")):
        print(ln[:150])
print(f"exit {r.returncode}")

reset    board rebooted (SEQ and ts_us restarted): epoch 1, run in progress discarded, dt state cleared  | valid=2956 bad=18 seq_breaks=27 lost=28 res
source=file valid=5907 malformed=1 empty=0 bad_checksum=29 seq_breaks=63 lost=64 loss=1.07% timing_breaks=0 resets=1 status=1 source_drops=0 capture_s
stances completed=60 discarded>MAX_DURATION(200)=0 discarded@reset=0 rejected<MIN_DURATION(15)=0 predicted=60 no_prediction=0 stances_with_gaps=34
FAIL: corrupted frames present
exit 1


## What is real and what is not

Real: the eight `_02` and `_03` captures, the sensor positions (±15 mm), the gain match and the bench measurements behind it, the stance counts and the classifier results in `docs/real_results.md`, the stage-14 bench captures in `data/bench/`, and the failure evidence in the `_01` set. Simulated: every `demo_*` stream here, the 12 bake-off sessions, and therefore every number in section 7's first block.

Still open: more sessions and a second subject (two sessions per class is the minimum for a per-session split, not a margin), the weight-shift activity that was never collected, and why BLE stalls when the board is powered from the PC's USB cable (the bench passed on battery).

In [13]:
print(f"notebook runtime: {time.time() - T_START:.1f} s")

notebook runtime: 38.2 s
